## **OSEMN | Obtain: Data Validation for Integrated IPEDS-CDS Dataset**
Validation checks are done to ensure that the join had been performed correctly. This included verifying the dataset structure, confirming the presence of all years (2019-2023) and ensuring that key admissions and enrollment variables such as applicants, admissions and enrollment counts were properly populated.  

**1. Environment Setup: Mount Drive to Upload in Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2. Dataset Loading and Initial Structure Review**

The integrated IPEDS-CDS dataset is loaded to examine its overall structure after data integration. This step verifies the number of rows and columns before performing detailed validation checks.


In [ ]:
import pandas as pd

#load dataset
df = pd.read_excel('/content/drive/MyDrive/P2 MDS/02_Data_Integration/IPEDS+CDS_final_dataset.xlsx')

#check number of rows and columns
print("Shape of dataset:", df.shape)

Shape of dataset: (367, 52)


**2.1 Observation Based on the Dataset Shape**

The dataset consists of 367 rows and 52 columns.
Based on the study design, the dataset is expected to contain:
60 institutions
5 years of data (2019-2023)
Expected total rows: 60 x 5 = 300 rows
However, the dataset contains 367 rows, indicating the presence of additional or inconsistent records.

In [ ]:
#check available years in dataset
print("Available years in dataset:")
print(sorted(df["year"].dropna().unique()))

#count records per year
print("\nNumber of records per year:")
print(df["year"].value_counts().sort_index())

Available years in dataset:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Number of records per year:
year
2019    60
2020    60
2021    61
2022    61
2023    61
2024    46
2025    18
Name: count, dtype: int64


In [ ]:
#institutions with 2024 data
inst_2024 = df[df["year"] == 2024]["institution_name"].unique()

#institutions with 2025 data
inst_2025 = df[df["year"] == 2025]["institution_name"].unique()

print("Number of institutions with 2024 data:", len(inst_2024))
print("Number of institutions with 2025 data:", len(inst_2025))

print("\nInstitutions with 2024 data:")
print(sorted(inst_2024))

print("\nInstitutions with 2025 data:")
print(sorted(inst_2025))

Number of institutions with 2024 data: 1
Number of institutions with 2025 data: 1

Institutions with 2024 data:
[nan]

Institutions with 2025 data:
[nan]


In [ ]:
#compare expected vs actual institutions
total_institutions = df["institution_name"].nunique()

print("Total institutions in dataset:", total_institutions)
print("2024 coverage:", len(inst_2024) / total_institutions)
print("2025 coverage:", len(inst_2025) / total_institutions)

Total institutions in dataset: 60
2024 coverage: 0.016666666666666666
2025 coverage: 0.016666666666666666


**2.2 Exclusion of Out of Scope Records**

2024-2025 records were excluded from the dataset as it is not intended for this study.

In [ ]:
#drop unwanted years
df = df[~df["year"].isin([2024, 2025])].copy()

In [ ]:
#confirm removal
print("Year distribution after filtering:")
print(df["year"].value_counts().sort_index())

Year distribution after filtering:
year
2019    60
2020    60
2021    61
2022    61
2023    61
Name: count, dtype: int64


In [ ]:
print("Before filtering:")
print(df["year"].value_counts().sort_index())

Before filtering:
year
2019    60
2020    60
2021    61
2022    61
2023    61
Name: count, dtype: int64


In [ ]:
print("After filtering:")
print(df["year"].value_counts().sort_index())

After filtering:
year
2019    60
2020    60
2021    61
2022    61
2023    61
Name: count, dtype: int64


In [ ]:
#save cleaned dataset
output_path = "/content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_2019_2023.xlsx"

df.to_excel(output_path, index=False)

print("File saved at:", output_path)

File saved at: /content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_2019_2023.xlsx


In [ ]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Shape:", df.shape)

Shape: (303, 52)


**3. Duplicates based on Institution and Year**

Identify duplicates based on institution and year based on the combination of institution identifier (UNITID) and year, as each institution should only have one record per year.


In [ ]:
#check duplicates based on institution + year
duplicates = df[df.duplicated(subset=["unitid", "year"], keep=False)]

print("Number of duplicate rows:", duplicates.shape[0])
duplicates.head()

Number of duplicate rows: 6


,unitid,institution_name,year,level_of_student,enrollment_men,enrollment_women,idx_ef,enrollment_asian,enrollment_black,enrollment_hispanic,...,graduation_women_Bachelor's or equiv subcohort (4-yr institution) Completers of bachelor's or equiv degrees total (150% of normal time),graduation_women_Bachelor's or equiv subcohort (4-yr institution) Transfer-out students,university,filename,pdf_path,retention_rate_y,applicants,admitted,enrolled,status
227,190150,Columbia University in the City of New York,2021,All students total,15638.0,18139.0,-2.0,4572.0,1780.0,2862.0,...,669.0,6.0,COLUMBIA UNIVERSITY,ColumbiaUniversity_CDS_2021-2022 (COLUMBIA COL...,/content/drive/MyDrive/cds_project/COLUMBIA UN...,NaN,60551.0,2355.0,1560.0,review_needed
228,190150,Columbia University in the City of New York,2021,All students total,15638.0,18139.0,-2.0,4572.0,1780.0,2862.0,...,669.0,6.0,COLUMBIA UNIVERSITY,ColumbiaUniversity_CDS_2021-2022 (COLUMBIA GEN...,/content/drive/MyDrive/cds_project/COLUMBIA UN...,NaN,540.0,150.0,NaN,review_needed
229,190150,Columbia University in the City of New York,2022,All students total,16534.0,19250.0,-2.0,4652.0,1833.0,3041.0,...,730.0,9.0,COLUMBIA UNIVERSITY,ColumbiaUniversity_CDS_2022-2023 (COLUMBIA COL...,/content/drive/MyDrive/cds_project/COLUMBIA UN...,NaN,60374.0,2255.0,1463.0,review_needed
230,190150,Columbia University in the City of New York,2022,All students total,16534.0,19250.0,-2.0,4652.0,1833.0,3041.0,...,730.0,9.0,COLUMBIA UNIVERSITY,ColumbiaUniversity_CDS_2022-2023 (COLUMBIA GEN...,/content/drive/MyDrive/cds_project/COLUMBIA UN...,NaN,504.0,149.0,NaN,review_needed
231,190150,Columbia University in the City of New York,2023,All students total,16128.0,19151.0,-2.0,4657.0,1899.0,3065.0,...,718.0,6.0,COLUMBIA UNIVERSITY,ColumbiaUniversity_CDS_2023-2024 (COLUMBIA COL...,/content/drive/MyDrive/cds_project/COLUMBIA UN...,NaN,57126.0,2284.0,1454.0,review_needed


**3.1 Identify Duplicates**

The analysis identified 6 duplicate rows within the dataset.
A closer inspection revealed that these duplicates occur for the same institution and year, with identical values across all variables. Columbia University in the City of New York appear more than once for the years 2021, 2022 and 2023 with exactly the same data.

In [ ]:
#count duplicates per institution-year
dup_counts = (
    df.groupby(["unitid", "year"])
      .size()
      .reset_index(name="count")
)

#show only duplicates
dup_counts = dup_counts[dup_counts["count"] > 1]

print("Duplicate institution-year combinations:")
print(dup_counts)

Duplicate institution-year combinations:
     unitid  year  count
187  190150  2021      2
188  190150  2022      2
189  190150  2023      2


In [ ]:
#example: inspect one duplicated institution-year
example = dup_counts.iloc[0]

df[
    (df["unitid"] == example["unitid"]) &
    (df["year"] == example["year"])
]

,unitid,institution_name,year,level_of_student,enrollment_men,enrollment_women,idx_ef,enrollment_asian,enrollment_black,enrollment_hispanic,...,graduation_women_Bachelor's or equiv subcohort (4-yr institution) Completers of bachelor's or equiv degrees total (150% of normal time),graduation_women_Bachelor's or equiv subcohort (4-yr institution) Transfer-out students,university,filename,pdf_path,retention_rate_y,applicants,admitted,enrolled,status
227,190150,Columbia University in the City of New York,2021,All students total,15638.0,18139.0,-2.0,4572.0,1780.0,2862.0,...,669.0,6.0,COLUMBIA UNIVERSITY,ColumbiaUniversity_CDS_2021-2022 (COLUMBIA COL...,/content/drive/MyDrive/cds_project/COLUMBIA UN...,NaN,60551.0,2355.0,1560.0,review_needed
228,190150,Columbia University in the City of New York,2021,All students total,15638.0,18139.0,-2.0,4572.0,1780.0,2862.0,...,669.0,6.0,COLUMBIA UNIVERSITY,ColumbiaUniversity_CDS_2021-2022 (COLUMBIA GEN...,/content/drive/MyDrive/cds_project/COLUMBIA UN...,NaN,540.0,150.0,NaN,review_needed


**3.2 Fix Duplicates**

In [ ]:
#count non-null values per row
df["non_null"] = df.notna().sum(axis=1)

#sort by completeness (descending)
df = df.sort_values("non_null", ascending=False)

#drop duplicates, keep most complete
df = df.drop_duplicates(subset=["unitid", "year"], keep="first")

#remove helper column
df = df.drop(columns="non_null")

print("Duplicates removed. New shape:", df.shape)

Duplicates removed. New shape: (300, 52)


**3.3 Validate Duplicates Fix**

In [ ]:
#check again
check = df.duplicated(subset=["unitid", "year"]).sum()
print("Remaining duplicate rows:", check)

Remaining duplicate rows: 0


In [ ]:
#save cleaned dataset
output_path = "//content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/finalvalidated_dataset_2019_2023.xlsx"

df.to_excel(output_path, index=False)

print("File saved at:", output_path)

File saved at: //content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/finalvalidated_dataset_2019_2023.xlsx
